# 第 3 课：让 AI 提取结构化信息 — JSON 输出与 Prompt Engineering

> 学完这节课，你会：让 AI 输出规范的 JSON、掌握 Prompt Engineering 核心技巧、从聊天记录中提取人物画像和关系小结。

---

## 3.1 为什么需要结构化输出

上节课我们学会了和 AI 对话，AI 返回的是自然语言文本。但在真实系统中，**程序需要的是结构化数据**。

类比前端开发：

```
用户在表单里填写信息 → 提交时转成 JSON → 发给后端 API
AI 从聊天记录里理解信息 → 输出 JSON → 存入数据库
```

你不可能让用户随便写一段话，然后直接存数据库。同样，你也不能让 AI 随便输出一段话，然后直接用 `JSON.parse()` 解析。

**在你们的联系人记忆系统中：**
- 用户截图聊天记录 → OCR 识别出文本
- AI 需要从文本中提取：联系人姓名、关系、偏好、聊天事实
- 这些信息必须是结构化的 JSON，才能存到 Core Data 里

所以，让 AI 稳定输出 JSON 是整个系统的基础。

## 3.2 让 AI 输出 JSON（基础）

先装好依赖、加载环境：

In [ ]:
!pip install openai python-dotenv -q

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"API Key 已加载（前8位: {api_key[:8]}...）")
else:
    print("未找到 OPENAI_API_KEY，请在课程目录创建 .env 文件")

### 方法一：在 prompt 里要求输出 JSON

最简单的方式——直接告诉 AI "请输出 JSON":

In [ ]:
chat_text = """小王：周末去爬香山吗？
我：好啊，上次爬得挺爽的
小王：这次叫上老李吧，他最近压力大
我：行，他喜欢户外，正好放松一下
小王：那我带咖啡，老李不是喜欢拿铁吗
我：对对，你记性真好"""

# 只在 prompt 里说"请输出 JSON"
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "你是一个信息提取助手。请从聊天记录中提取提到的联系人信息，输出 JSON 格式。"
        },
        {
            "role": "user",
            "content": f"请从以下聊天记录中提取联系人信息：\n\n{chat_text}"
        }
    ]
)

result = response.choices[0].message.content
print("AI 的原始输出：")
print(result)
print("\n---")
print("类型:", type(result))  # 注意：这是 str，不是 dict

问题来了：AI 输出的可能是：
- 混了一些解释文字（"以下是提取的信息：..."）
- 被 Markdown 代码块包裹（\`\`\`json ... \`\`\`）
- 格式不稳定，每次调用结构可能不一样

直接 `json.loads()` 大概率会报错。

### 方法二：使用 response_format 参数（推荐）

OpenAI 提供了 `response_format` 参数，**强制 AI 输出合法的 JSON**：

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},  # 关键参数！
    messages=[
        {
            "role": "system",
            "content": """你是一个信息提取助手。请从聊天记录中提取提到的联系人信息。
输出 JSON 格式，包含一个 contacts 数组，每个联系人有以下字段：
- name: 姓名
- relationship: 与用户的关系
- preferences: 已知的偏好列表"""
        },
        {
            "role": "user",
            "content": f"请从以下聊天记录中提取联系人信息：\n\n{chat_text}"
        }
    ]
)

result = response.choices[0].message.content
print("AI 输出的原始字符串：")
print(result)

# 现在可以安全地解析 JSON 了
data = json.loads(result)
print("\n解析后的 Python 对象：")
print(json.dumps(data, ensure_ascii=False, indent=2))
print(f"\n提取到 {len(data.get('contacts', []))} 个联系人")

### 对比

| 方式 | 可靠性 | 说明 |
|------|--------|------|
| 只在 prompt 里说 | 低 | AI 可能加解释文字、格式不稳定 |
| `response_format={"type": "json_object"}` | 高 | API 层面保证输出是合法 JSON |

> **注意：** 使用 `response_format` 时，system prompt 里**必须提到 JSON**，否则 API 会报错。

---

## 3.3 Prompt Engineering 技巧

让 AI 输出 JSON 只是第一步。更重要的是：**怎么写 prompt 才能让输出质量更高？**

下面介绍三个核心技巧。

### 技巧一：Few-shot（给示例）

"Talk is cheap, show me the example." 与其描述你想要什么格式，不如直接给 AI 一个例子。

In [ ]:
# 不给示例（Zero-shot）
response_zero = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "从聊天记录提取联系人信息，输出 JSON。"
        },
        {
            "role": "user",
            "content": "小张：下周二我生日，来我家吃火锅吧\n我：好啊！你喜欢什么口味的蛋糕？\n小张：巧克力的！"
        }
    ]
)

print("【Zero-shot 输出】（没给示例）")
print(json.dumps(json.loads(response_zero.choices[0].message.content), ensure_ascii=False, indent=2))

In [ ]:
# 给示例（Few-shot）
response_few = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": """从聊天记录提取联系人信息，输出 JSON。

示例输入：
老王：明天去打篮球吗
我：好啊，叫上小刘
老王：行，我带水

示例输出：
{
  "contacts": [
    {
      "name": "老王",
      "facts": ["喜欢打篮球", "会主动组织活动"],
      "upcoming_events": ["明天一起打篮球"]
    },
    {
      "name": "小刘",
      "facts": ["认识老王", "可能也喜欢打篮球"],
      "upcoming_events": []
    }
  ]
}

请严格按照示例中的 JSON 格式输出。"""
        },
        {
            "role": "user",
            "content": "小张：下周二我生日，来我家吃火锅吧\n我：好啊！你喜欢什么口味的蛋糕？\n小张：巧克力的！"
        }
    ]
)

print("【Few-shot 输出】（给了示例）")
print(json.dumps(json.loads(response_few.choices[0].message.content), ensure_ascii=False, indent=2))

对比两个输出：Few-shot 版本的字段命名、结构层次都更可控。

**核心原则：** 给 AI 一个"标准答案"，它就会模仿着来。

### 技巧二：明确指定 Schema

比 few-shot 更严格的方式——直接告诉 AI 输出的 JSON Schema：

In [ ]:
schema_prompt = """你是一个聊天记录分析助手。请提取联系人信息，严格按照以下 JSON Schema 输出：

{
  "contacts": [
    {
      "name": "string, 联系人姓名",
      "relationship": "string, 与用户的关系，如：朋友/同事/家人/未知",
      "personality_traits": ["string, 性格特点"],
      "preferences": ["string, 已知偏好"],
      "facts": ["string, 从聊天中获取的事实"],
      "confidence": "number, 0-1 之间的置信度"
    }
  ]
}

规则：
- 只提取聊天中明确提到的信息，不要推测
- confidence 表示你对提取结果的确信程度
- 如果某个字段无法确定，用空数组或"未知""""

test_chat = """小李：哥，妈让我问你周末回不回家吃饭
我：回，让妈别做辣的，我最近胃不舒服
小李：好的，那我让妈炖个排骨汤
我：你不是要考试吗？复习得怎么样
小李：别提了，高数太难了"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": schema_prompt},
        {"role": "user", "content": f"请分析以下聊天记录：\n\n{test_chat}"}
    ]
)

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, ensure_ascii=False, indent=2))

### 技巧三：Chain of Thought（分步思考）

让 AI 先"想一想"再输出结果，可以提高提取准确率。

类比：你面试时遇到复杂问题，先在纸上理一理思路，再回答，效果会更好。

In [ ]:
cot_prompt = """你是一个聊天记录分析助手。

请按以下步骤处理：

第一步：识别聊天中出现的所有人物，列出他们的名字
第二步：对每个人物，逐条分析聊天记录，提取关于他的事实
第三步：根据事实推断人物关系和性格特点
第四步：输出最终的 JSON 结果

输出 JSON 格式：
{
  "thinking": "你的分析过程（第一步到第三步的思考）",
  "contacts": [
    {
      "name": "姓名",
      "relationship": "关系",
      "personality_traits": ["性格特点"],
      "facts": ["事实"]
    }
  ]
}"""

complex_chat = """小美：你帮我问问老陈明天有没有空，我们部门团建
我：老陈最近忙得很，天天加班
小美：哎，他也太拼了，上次聚餐他也没来
我：他就是那种工作狂的性格
小美：那算了，我直接叫阿伟吧，他肯定来，他最爱吃烧烤了
我：行，阿伟那个吃货肯定秒回"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": cot_prompt},
        {"role": "user", "content": f"请分析以下聊天记录：\n\n{complex_chat}"}
    ]
)

data = json.loads(response.choices[0].message.content)

# 先看 AI 的思考过程
print("=== AI 的思考过程 ===")
print(data.get("thinking", "无"))

# 再看最终结果
print("\n=== 提取的联系人 ===")
for contact in data.get("contacts", []):
    print(f"\n  {contact['name']}")
    print(f"    关系: {contact.get('relationship', '未知')}")
    print(f"    性格: {', '.join(contact.get('personality_traits', []))}")
    print(f"    事实: {contact.get('facts', [])}")

### 三个技巧总结

| 技巧 | 适用场景 | 效果 |
|------|---------|------|
| Few-shot | 需要 AI 模仿特定格式 | 输出格式稳定 |
| Schema 约束 | 需要严格的数据结构 | 字段命名一致 |
| Chain of Thought | 复杂的信息提取任务 | 提取准确率更高 |

在实际项目中，通常**三个技巧组合使用**。

---

## 3.4 实战：从聊天记录提取人物画像

现在我们来模拟你们系统 `ContactDetectionService` 做的事情：

```
用户截图聊天记录 → OCR 识别出文本 → AI 提取联系人信息 → 存入数据库
```

我们跳过 OCR 那一步，直接用文本聊天记录作为输入。

In [ ]:
# 模拟微信聊天记录（假设 OCR 已经识别出来了）
wechat_chat = """李雪琴 2024-03-15 10:23
在吗？我想问你个事

我 2024-03-15 10:25
在在在，怎么了

李雪琴 2024-03-15 10:26
上次你推荐的那个日料店叫什么来着？我想带我妈去吃

我 2024-03-15 10:27
鮨一！在国贸那边，他们家三文鱼特别好

李雪琴 2024-03-15 10:27
对对，就是那家。你说他们周末要排队是吧

我 2024-03-15 10:28
周末排死了，建议工作日去。你妈不是退休了吗，周三周四随便去

李雪琴 2024-03-15 10:29
有道理！对了，下周六你有空吗？公司周年庆结束后想约你和陈磊吃饭

我 2024-03-15 10:30
可以啊，陈磊最近刚从日本出差回来，正好让他推荐点清酒

李雪琴 2024-03-15 10:31
哈哈他肯定又带了一堆零食回来。那就这么定了！

李雪琴 2024-03-15 10:32
话说你们技术部最近加班多吗？我们产品这边需求排到下个月了

我 2024-03-15 10:33
还行吧，主要是老板要的那个 AI 功能比较赶"""

print("聊天记录长度:", len(wechat_chat), "字符")
print("预览前 200 字：")
print(wechat_chat[:200] + "...")

In [ ]:
# 这就是你们系统 ContactDetectionService 的核心 prompt
detection_prompt = """你是一个聊天记录分析引擎。你的任务是从 OCR 识别出的聊天记录中提取结构化信息。

请严格按照以下 JSON 格式输出：

{
  "detected_contacts": [
    {
      "detected_name": "在聊天中出现的名字",
      "platform_hint": "聊天平台（微信/iMessage/WhatsApp/未知）",
      "msg_history": [
        {
          "sender": "发送者名字",
          "content": "消息内容",
          "timestamp": "时间戳（如有）"
        }
      ],
      "personality_traits": ["从聊天中推断出的性格特点"],
      "extracted_facts": ["从聊天中提取的客观事实"],
      "relationship_to_user": "与用户的关系推断"
    }
  ],
  "chat_context": {
    "platform": "聊天平台",
    "date_range": "聊天时间范围",
    "main_topics": ["主要话题"]
  }
}

规则：
1. detected_name 只填聊天中明确出现的名字，不要猜测真名
2. msg_history 只包含与该联系人直接相关的消息
3. personality_traits 需要有聊天内容支撑，不要凭空推断
4. extracted_facts 只写聊天中明确提到的事实
5. 如果聊天中提到了第三方人物（不是直接参与聊天的），也要提取"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    temperature=0,  # 信息提取任务用低温度，保证稳定性
    messages=[
        {"role": "system", "content": detection_prompt},
        {"role": "user", "content": f"请分析以下聊天记录：\n\n{wechat_chat}"}
    ]
)

detection_result = json.loads(response.choices[0].message.content)
print(json.dumps(detection_result, ensure_ascii=False, indent=2))

In [ ]:
# 遍历提取结果，模拟存入数据库的过程
print("=== 联系人检测结果 ===")
print(f"平台: {detection_result.get('chat_context', {}).get('platform', '未知')}")
print(f"时间: {detection_result.get('chat_context', {}).get('date_range', '未知')}")
print(f"话题: {detection_result.get('chat_context', {}).get('main_topics', [])}")
print()

for i, contact in enumerate(detection_result.get("detected_contacts", [])):
    print(f"--- 联系人 {i+1}: {contact['detected_name']} ---")
    print(f"  关系: {contact.get('relationship_to_user', '未知')}")
    print(f"  性格: {contact.get('personality_traits', [])}")
    print(f"  事实:")
    for fact in contact.get("extracted_facts", []):
        print(f"    - {fact}")
    print(f"  相关消息数: {len(contact.get('msg_history', []))}")
    print()

### 这就是你们系统在做的事

```
iOS App 截图 → OCR 识别文本 → 上面这段代码（AI 提取） → Core Data 存储
                                      ↑
                         ContactDetectionService
```

你们系统里 `ContactDetectionService` 的核心逻辑，本质上就是上面这个 prompt + JSON 解析。区别在于：
- 真实系统还要处理 OCR 的识别误差
- 需要和已有联系人做匹配（去重）
- 需要增量更新而不是覆盖

---

## 3.5 实战：生成关系小结

提取出联系人信息后，下一步是生成**人类可读的关系小结**。

这对应你们系统的 Memory Layer：把零散的事实浓缩成一段有用的描述。

```
原始事实 → AI 总结 → 关系小结 + 社交 Tips
```

In [ ]:
# 用上一步提取出的事实，生成关系小结
# 先把提取结果整理成输入
facts_input = []
for contact in detection_result.get("detected_contacts", []):
    facts_input.append({
        "name": contact["detected_name"],
        "relationship": contact.get("relationship_to_user", "未知"),
        "personality_traits": contact.get("personality_traits", []),
        "facts": contact.get("extracted_facts", [])
    })

print("输入的联系人事实：")
print(json.dumps(facts_input, ensure_ascii=False, indent=2))

In [ ]:
summary_prompt = """你是一个社交关系分析助手。根据用户的联系人信息，生成关系小结和社交建议。

输出 JSON 格式：
{
  "summaries": [
    {
      "contact_name": "联系人姓名",
      "relationship_summary": "一段 2-3 句话的关系描述，自然口语化",
      "social_tips": ["具体可操作的社交建议"],
      "conversation_starters": ["下次聊天可以用的开场白"]
    }
  ],
  "overall_insight": "对用户社交圈的整体洞察（1-2 句话）"
}

要求：
- relationship_summary 要自然，像朋友在给你介绍一样
- social_tips 要具体、可操作，不要说空话
- conversation_starters 要基于已知事实，自然不尴尬"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    temperature=0.7,  # 生成建议类内容，稍微提高创意度
    messages=[
        {"role": "system", "content": summary_prompt},
        {
            "role": "user",
            "content": f"请根据以下联系人信息生成关系小结：\n\n{json.dumps(facts_input, ensure_ascii=False)}"
        }
    ]
)

summary_result = json.loads(response.choices[0].message.content)
print(json.dumps(summary_result, ensure_ascii=False, indent=2))

In [ ]:
# 格式化展示（模拟 App 中的展示效果）
print("=" * 50)
print("    联系人关系小结")
print("=" * 50)

for s in summary_result.get("summaries", []):
    print(f"\n【{s['contact_name']}】")
    print(f"  {s['relationship_summary']}")
    print(f"\n  社交 Tips:")
    for tip in s.get("social_tips", []):
        print(f"    - {tip}")
    print(f"\n  下次可以聊:")
    for starter in s.get("conversation_starters", []):
        print(f"    > {starter}")

print(f"\n{'=' * 50}")
print(f"整体洞察: {summary_result.get('overall_insight', '')}")
print(f"{'=' * 50}")

### 这就是 Memory Layer 在做的事

你们系统的 Memory Layer 工作流程：

```
ContactDetectionService 提取的事实
         ↓
Memory Layer（上面这段代码）
         ↓
关系小结 + 社交 Tips
         ↓
注入到回复生成的 System Prompt 中
         ↓
AI 生成更有温度的回复建议
```

注意两次 AI 调用的 `temperature` 不同：
- 提取事实用 `temperature=0`（需要准确、稳定）
- 生成小结用 `temperature=0.7`（需要自然、有变化）

---

## 3.6 本课小结

| 概念 | 你学到了什么 |
|------|----------|
| 结构化输出 | 用 `response_format` 让 AI 稳定输出 JSON |
| Few-shot | 给示例比给描述更有效 |
| Schema 约束 | 明确指定字段名和类型，输出更可控 |
| Chain of Thought | 让 AI 先思考再输出，准确率更高 |
| 信息提取 | 从非结构化文本中提取结构化数据 |
| 关系小结 | 把零散事实浓缩成可读的描述和建议 |

### 对应你们系统的模块

```
本课内容                    你们系统模块
─────────────────────────────────────────
3.2 JSON 输出基础      →   所有 AI 调用的基础
3.3 Prompt Engineering →   各个 Service 的 prompt 设计
3.4 提取人物画像       →   ContactDetectionService
3.5 生成关系小结       →   Memory Layer
```

### 关键认识

**Prompt 就是你的"产品需求文档"。** 写给 AI 的 prompt 和写给开发者的 PRD 一样——越清晰、越具体，输出质量越高。模糊的 prompt 就像模糊的需求，结果一定不会好。

### 下节预告

下一课我们来处理一个现实问题：**聊天记录可能很长，超过 AI 的处理能力怎么办？** 我们将学习文本分割和上下文窗口管理——这就是你们系统处理长截图的方案。